# Optimización de turnos y cobertura: preparación

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("workforce_scheduling")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from workforce_scheduling.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from workforce_scheduling.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"preparacion")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Optimización de turnos y cobertura: preparacion

Escenario de **14 empleados ficticios** y **140 franjas** de una semana hipotética. Demanda requerida: **297.5 agente-horas**. Las entradas son **sintéticas y locales**, claramente identificadas; no se generan pronósticos, llamadas ni requisitos mediante Erlang C. La optimización consume requisitos de cobertura ya definidos y no depende de otros proyectos.


,Indicador,Valor
0,origen,Escenario didáctico sintético; no datos de ope...
1,fuente_sha256,38aa8e71618f2b91f1268f4852e44879fc2b4bca968884...
2,empleados_sha256,4807349031c2fa5563bc69d8cf43e166fa6dc893229ff2...
3,empleados,14
4,intervalos,140
5,demanda_agente_horas,297.5
6,particiones,"No aplica: optimización de un escenario fijo, ..."


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from workforce_scheduling.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

[Abrir informe integral](../reports/informe_final/informe_final.md)